# UAVDT Notebook 06 — Validate Frame Mapping, Detections, BEV Projection, and Tracks

This notebook diagnoses why moving 3D cuboids may not correspond to the real vehicles.

It creates:

- frame-to-image mapping tables
- original-frame detection overlays
- BEV projection overlays
- side-by-side original + BEV diagnostic frames/video
- per-track sanity metrics

Run after Notebooks 02–05.


In [ ]:
#@title 1. Set local project paths
from pathlib import Path
import json
import math
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from notebook_local import resolve_project_dir, print_local_setup

try:
    from IPython.display import display, Image, HTML
except Exception:
    pass

PROJECT_DIR = resolve_project_dir()
WORK_DIR = PROJECT_DIR / 'work'
SAMPLE_DIR = WORK_DIR / 'M1401_sample'
SAMPLE_IMAGES_DIR = SAMPLE_DIR / 'images'
NB02_DIR = WORK_DIR / 'notebook_02_bev_homography'
NB03_DIR = WORK_DIR / 'notebook_03_bev_vehicles'
NB04_DIR = WORK_DIR / 'notebook_04_metric_scene_export'
NB05_DIR = WORK_DIR / 'notebook_05_dynamic_scene_visualization'
NB06_DIR = WORK_DIR / 'notebook_06_validation'
NB06_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('SAMPLE_IMAGES_DIR:', SAMPLE_IMAGES_DIR, 'exists:', SAMPLE_IMAGES_DIR.exists())
print('NB02_DIR:', NB02_DIR, 'exists:', NB02_DIR.exists())
print('NB03_DIR:', NB03_DIR, 'exists:', NB03_DIR.exists())
print('NB04_DIR:', NB04_DIR, 'exists:', NB04_DIR.exists())
print('NB06_DIR:', NB06_DIR)


In [ ]:
#@title 2. Load sample images and reconstruct sample frame mapping

image_exts = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
sample_images = []
for pat in image_exts:
    sample_images.extend(SAMPLE_IMAGES_DIR.glob(pat))
sample_images = sorted(sample_images)

if len(sample_images) == 0:
    raise FileNotFoundError('No sample images found in ' + str(SAMPLE_IMAGES_DIR))

print('Sample images found:', len(sample_images))
print('First 10 sample images:')
for i, p in enumerate(sample_images[:10]):
    print(i, p.name)

# Try to find an explicit mapping saved by earlier notebooks.
possible_mapping_files = [
    SAMPLE_DIR / 'frame_mapping.csv',
    SAMPLE_DIR / 'sample_frame_mapping.csv',
    WORK_DIR / 'M1401_sample_frame_mapping.csv',
]

mapping_path = None
for p in possible_mapping_files:
    if p.exists():
        mapping_path = p
        break

if mapping_path is not None:
    frame_mapping = pd.read_csv(mapping_path)
    print('Loaded existing frame mapping:', mapping_path)
else:
    rows = []
    for i, p in enumerate(sample_images):
        rows.append({
            'frame_id': i,
            'sample_image': p.name,
            'sample_image_path': str(p),
            'original_image': None,
            'original_image_path': None,
            'mapping_source': 'sample_order_only',
        })
    frame_mapping = pd.DataFrame(rows)
    mapping_path = NB06_DIR / 'frame_mapping_reconstructed.csv'
    frame_mapping.to_csv(mapping_path, index=False)
    print('No explicit mapping found. Reconstructed by sample image order only.')
    print('Saved:', mapping_path)

print('Frame mapping columns:', frame_mapping.columns.tolist())
display(frame_mapping.head(20))


In [ ]:
#@title 3. Load homography from Notebook 02

candidate_paths = []
candidate_paths += [NB02_DIR / 'homography.json', NB02_DIR / 'bev_homography.json']
candidate_paths += sorted(NB02_DIR.glob('*homography*.json'))

print('Candidate homography files:')
for p in candidate_paths:
    print(' ', p, 'exists:', p.exists())

homography_path = None
for p in candidate_paths:
    if p.exists():
        homography_path = p
        break

if homography_path is None:
    raise FileNotFoundError('Could not find homography JSON in ' + str(NB02_DIR))

with open(homography_path, 'r') as f:
    homography_data = json.load(f)

if 'H_image_to_bev' in homography_data:
    H_image_to_bev = np.array(homography_data['H_image_to_bev'], dtype=np.float32)
elif 'homography' in homography_data:
    H_image_to_bev = np.array(homography_data['homography'], dtype=np.float32)
elif 'H' in homography_data:
    H_image_to_bev = np.array(homography_data['H'], dtype=np.float32)
else:
    raise KeyError('Homography matrix not found in JSON. Keys: ' + str(list(homography_data.keys())))

BEV_WIDTH = int(homography_data.get('bev_width', homography_data.get('BEV_WIDTH', 600)))
BEV_HEIGHT = int(homography_data.get('bev_height', homography_data.get('BEV_HEIGHT', 1000)))

print('Using homography:', homography_path)
print('BEV size:', BEV_WIDTH, BEV_HEIGHT)
print('H_image_to_bev:')
print(H_image_to_bev)


In [ ]:
#@title 4. Load detections and/or metric tracks

def list_csvs(path):
    if not path.exists():
        return []
    return sorted(path.glob('*.csv'))

print('CSV files in Notebook 03:')
for p in list_csvs(NB03_DIR):
    print(' ', p.name)

print('CSV files in Notebook 04:')
for p in list_csvs(NB04_DIR):
    print(' ', p.name)

# Detection candidates from Notebook 03.
detection_candidates = [
    NB03_DIR / 'detections.csv',
    NB03_DIR / 'vehicle_detections.csv',
    NB03_DIR / 'vehicle_detections_yolo.csv',
    NB03_DIR / 'detections_projected_bev.csv',
    NB03_DIR / 'vehicle_tracks_bev.csv',
]
detection_candidates += [p for p in list_csvs(NB03_DIR) if any(k in p.name.lower() for k in ['detect', 'vehicle', 'track', 'bev'])]

detections_path = None
for p in detection_candidates:
    if p.exists():
        detections_path = p
        break

if detections_path is None:
    raise FileNotFoundError('Could not find detection CSV in ' + str(NB03_DIR))

detections = pd.read_csv(detections_path)
print('Loaded detections/tracks candidate:', detections_path)
print('Shape:', detections.shape)
print('Columns:', detections.columns.tolist())
display(detections.head())

# Metric tracks from Notebook 04, if available.
metric_candidates = [
    NB04_DIR / 'vehicle_tracks_metric.csv',
    NB04_DIR / 'vehicle_tracks_metric_normalized.csv',
]
metric_path = None
for p in metric_candidates:
    if p.exists():
        metric_path = p
        break

if metric_path is not None:
    metric_tracks = pd.read_csv(metric_path)
    print('Loaded metric tracks:', metric_path)
    print('Metric shape:', metric_tracks.shape)
    print('Metric columns:', metric_tracks.columns.tolist())
    display(metric_tracks.head())
else:
    metric_tracks = None
    print('No metric tracks found. Validation will use detections and BEV projection only.')


In [ ]:
#@title 5. Normalize detections, project to BEV, and ensure track IDs

# This cell creates validation_tracks with image boxes, BEV coordinates, frame_id, and track_id.

def find_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError('Could not find any of ' + str(candidates) + '. Columns: ' + str(df.columns.tolist()))
    return None

frame_col = find_col(detections, ['frame_id', 'frame_idx', 'frame_index', 'frame'])
if frame_col == 'frame' and detections[frame_col].dtype == object:
    # Keep filename separately and use order mapping if no numeric frame column exists.
    detections = detections.copy()
    detections['frame_name'] = detections['frame']
    name_to_id = {p.name: i for i, p in enumerate(sample_images)}
    detections['frame_id'] = detections['frame_name'].map(name_to_id)
    if detections['frame_id'].isna().any():
        detections['frame_id'] = pd.factorize(detections['frame_name'])[0]
    frame_col = 'frame_id'

x1_col = find_col(detections, ['x1', 'xmin', 'left'])
y1_col = find_col(detections, ['y1', 'ymin', 'top'])
x2_col = find_col(detections, ['x2', 'xmax', 'right'])
y2_col = find_col(detections, ['y2', 'ymax', 'bottom'])
class_col = find_col(detections, ['class_name', 'class', 'label'], required=False)
conf_col = find_col(detections, ['confidence', 'conf', 'score'], required=False)
track_col = find_col(detections, ['track_id', 'id', 'object_id', 'track'], required=False)

validation_tracks = detections.copy()
validation_tracks['frame_id'] = pd.to_numeric(validation_tracks[frame_col], errors='coerce').astype(int)
validation_tracks['x1'] = pd.to_numeric(validation_tracks[x1_col], errors='coerce')
validation_tracks['y1'] = pd.to_numeric(validation_tracks[y1_col], errors='coerce')
validation_tracks['x2'] = pd.to_numeric(validation_tracks[x2_col], errors='coerce')
validation_tracks['y2'] = pd.to_numeric(validation_tracks[y2_col], errors='coerce')
validation_tracks['class_name'] = validation_tracks[class_col].astype(str) if class_col else 'vehicle'
validation_tracks['confidence'] = pd.to_numeric(validation_tracks[conf_col], errors='coerce') if conf_col else np.nan

# Project bottom-center unless existing BEV columns are already present.
if 'bev_x_px' in validation_tracks.columns and 'bev_y_px' in validation_tracks.columns:
    validation_tracks['bev_x_px'] = pd.to_numeric(validation_tracks['bev_x_px'], errors='coerce')
    validation_tracks['bev_y_px'] = pd.to_numeric(validation_tracks['bev_y_px'], errors='coerce')
else:
    validation_tracks['bbox_cx'] = 0.5 * (validation_tracks['x1'] + validation_tracks['x2'])
    validation_tracks['bbox_bottom_y'] = validation_tracks['y2']
    pts = validation_tracks[['bbox_cx', 'bbox_bottom_y']].to_numpy(dtype=np.float32).reshape(-1, 1, 2)
    bev_pts = cv2.perspectiveTransform(pts, H_image_to_bev).reshape(-1, 2)
    validation_tracks['bev_x_px'] = bev_pts[:, 0]
    validation_tracks['bev_y_px'] = bev_pts[:, 1]

# Keep a validity flag but do not drop yet.
validation_tracks['inside_bev'] = (
    (validation_tracks['bev_x_px'] >= 0) &
    (validation_tracks['bev_x_px'] <= BEV_WIDTH) &
    (validation_tracks['bev_y_px'] >= 0) &
    (validation_tracks['bev_y_px'] <= BEV_HEIGHT)
)

# Add or create track IDs. If no track IDs exist, use a simple diagnostic nearest-neighbor tracker.
if track_col is not None:
    validation_tracks['track_id'] = validation_tracks[track_col]
else:
    MAX_MATCH_DISTANCE_PX = 90.0
    MAX_MISSED_FRAMES = 3
    next_track_id = 1
    active = {}
    output_rows = []
    for frame_id, frame_df in validation_tracks.sort_values('frame_id').groupby('frame_id'):
        for tid in active:
            active[tid]['updated'] = False
        for _, det in frame_df.iterrows():
            bx = float(det['bev_x_px'])
            by = float(det['bev_y_px'])
            best_tid = None
            best_dist = float('inf')
            for tid, state in active.items():
                if frame_id - state['last_frame'] > MAX_MISSED_FRAMES:
                    continue
                if state.get('updated', False):
                    continue
                dist = math.hypot(bx - state['bev_x_px'], by - state['bev_y_px'])
                if dist < best_dist:
                    best_dist = dist
                    best_tid = tid
            if best_tid is None or best_dist > MAX_MATCH_DISTANCE_PX:
                best_tid = next_track_id
                next_track_id += 1
            row = det.to_dict()
            row['track_id'] = best_tid
            row['match_distance_px'] = best_dist if np.isfinite(best_dist) else np.nan
            output_rows.append(row)
            active[best_tid] = {'bev_x_px': bx, 'bev_y_px': by, 'last_frame': int(frame_id), 'updated': True}
        stale = [tid for tid, st in active.items() if frame_id - st['last_frame'] > MAX_MISSED_FRAMES]
        for tid in stale:
            del active[tid]
    validation_tracks = pd.DataFrame(output_rows)

validation_tracks = validation_tracks.sort_values(['frame_id', 'track_id']).reset_index(drop=True)
validation_path = NB06_DIR / 'validation_tracks_bev.csv'
validation_tracks.to_csv(validation_path, index=False)
print('Saved validation tracks:', validation_path)
print('Rows:', len(validation_tracks))
print('Unique frames:', validation_tracks['frame_id'].nunique())
print('Unique tracks:', validation_tracks['track_id'].nunique())
print('Inside BEV fraction:', float(validation_tracks['inside_bev'].mean()))
display(validation_tracks.head(20))


In [ ]:
#@title 6. Create original-frame detection overlays with track IDs

ORIGINAL_OVERLAY_DIR = NB06_DIR / 'original_detection_overlays'
ORIGINAL_OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

MAX_OVERLAY_FRAMES = 30 #@param {type:'integer'}
DRAW_CONFIDENCE = True #@param {type:'boolean'}

frame_ids = sorted(validation_tracks['frame_id'].unique().tolist())[:MAX_OVERLAY_FRAMES]

def color_for_id(track_id):
    rng = np.random.default_rng(int(track_id) * 9973 + 123)
    return tuple(int(v) for v in rng.integers(40, 255, size=3))

def get_sample_image_path(frame_id):
    if 0 <= int(frame_id) < len(sample_images):
        return sample_images[int(frame_id)]
    row = frame_mapping[frame_mapping['frame_id'] == int(frame_id)]
    if len(row) > 0 and 'sample_image_path' in row.columns:
        p = Path(str(row.iloc[0]['sample_image_path']))
        if p.exists():
            return p
    return None

overlay_paths = []
for frame_id in frame_ids:
    img_path = get_sample_image_path(frame_id)
    if img_path is None or not img_path.exists():
        print('Missing image for frame_id:', frame_id)
        continue
    img = cv2.imread(str(img_path))
    if img is None:
        print('Could not read image:', img_path)
        continue
    fdf = validation_tracks[validation_tracks['frame_id'] == frame_id]
    for _, r in fdf.iterrows():
        tid = int(r['track_id'])
        color = color_for_id(tid)
        x1, y1, x2, y2 = [int(round(float(r[c]))) for c in ['x1', 'y1', 'x2', 'y2']]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = 'ID ' + str(tid)
        if DRAW_CONFIDENCE and np.isfinite(float(r.get('confidence', np.nan))):
            label += ' ' + str(round(float(r['confidence']), 2))
        cv2.putText(img, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
        cx = int(round(0.5 * (float(r['x1']) + float(r['x2']))))
        by = int(round(float(r['y2'])))
        cv2.circle(img, (cx, by), 4, color, -1)
    out_path = ORIGINAL_OVERLAY_DIR / ('original_overlay_frame_' + str(int(frame_id)).zfill(6) + '.jpg')
    cv2.imwrite(str(out_path), img)
    overlay_paths.append(out_path)

print('Saved original overlays:', len(overlay_paths))
print('Directory:', ORIGINAL_OVERLAY_DIR)
if overlay_paths:
    display(Image(filename=str(overlay_paths[0])))


In [ ]:
#@title 7. Create BEV projection overlays with the same track IDs

BEV_OVERLAY_DIR = NB06_DIR / 'bev_projection_overlays'
BEV_OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

MAX_BEV_OVERLAY_FRAMES = 30 #@param {type:'integer'}
POINT_RADIUS = 5 #@param {type:'integer'}

frame_ids = sorted(validation_tracks['frame_id'].unique().tolist())[:MAX_BEV_OVERLAY_FRAMES]
bev_overlay_paths = []

for frame_id in frame_ids:
    img_path = get_sample_image_path(frame_id)
    if img_path is None or not img_path.exists():
        continue
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    bev = cv2.warpPerspective(img, H_image_to_bev, (BEV_WIDTH, BEV_HEIGHT))
    fdf = validation_tracks[validation_tracks['frame_id'] == frame_id]
    for _, r in fdf.iterrows():
        tid = int(r['track_id'])
        color = color_for_id(tid)
        x = int(round(float(r['bev_x_px'])))
        y = int(round(float(r['bev_y_px'])))
        if -50 <= x <= BEV_WIDTH + 50 and -50 <= y <= BEV_HEIGHT + 50:
            cv2.circle(bev, (x, y), POINT_RADIUS, color, -1)
            cv2.putText(bev, str(tid), (x + 6, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    out_path = BEV_OVERLAY_DIR / ('bev_overlay_frame_' + str(int(frame_id)).zfill(6) + '.jpg')
    cv2.imwrite(str(out_path), bev)
    bev_overlay_paths.append(out_path)

print('Saved BEV overlays:', len(bev_overlay_paths))
print('Directory:', BEV_OVERLAY_DIR)
if bev_overlay_paths:
    display(Image(filename=str(bev_overlay_paths[0])))


In [ ]:
#@title 8. Create side-by-side diagnostic frames and MP4 video

import imageio.v2 as imageio

SIDE_BY_SIDE_DIR = NB06_DIR / 'side_by_side_frames'
SIDE_BY_SIDE_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_FPS = 8 #@param {type:'integer'}
MAX_VIDEO_FRAMES = 30 #@param {type:'integer'}
OUTPUT_WIDTH = 1600 #@param {type:'integer'}

side_paths = []
for i, frame_id in enumerate(sorted(validation_tracks['frame_id'].unique().tolist())[:MAX_VIDEO_FRAMES]):
    left_path = ORIGINAL_OVERLAY_DIR / ('original_overlay_frame_' + str(int(frame_id)).zfill(6) + '.jpg')
    right_path = BEV_OVERLAY_DIR / ('bev_overlay_frame_' + str(int(frame_id)).zfill(6) + '.jpg')
    if not left_path.exists() or not right_path.exists():
        continue
    left = cv2.imread(str(left_path))
    right = cv2.imread(str(right_path))
    if left is None or right is None:
        continue
    # Resize both to the same height.
    target_h = 700
    left_w = int(left.shape[1] * target_h / left.shape[0])
    right_w = int(right.shape[1] * target_h / right.shape[0])
    left_r = cv2.resize(left, (left_w, target_h))
    right_r = cv2.resize(right, (right_w, target_h))
    side = np.concatenate([left_r, right_r], axis=1)
    if side.shape[1] > OUTPUT_WIDTH:
        new_h = int(side.shape[0] * OUTPUT_WIDTH / side.shape[1])
        side = cv2.resize(side, (OUTPUT_WIDTH, new_h))
    cv2.putText(side, 'Frame ' + str(int(frame_id)) + ' | Left: original detections | Right: BEV projection', (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    out_path = SIDE_BY_SIDE_DIR / ('side_by_side_frame_' + str(int(frame_id)).zfill(6) + '.jpg')
    cv2.imwrite(str(out_path), side)
    side_paths.append(out_path)

print('Saved side-by-side frames:', len(side_paths))
print('Directory:', SIDE_BY_SIDE_DIR)

if side_paths:
    video_path = NB06_DIR / 'validation_original_vs_bev.mp4'
    gif_path = NB06_DIR / 'validation_original_vs_bev.gif'
    imgs = [imageio.imread(p) for p in side_paths]
    imageio.mimsave(video_path, imgs, fps=VIDEO_FPS)
    imageio.mimsave(gif_path, imgs, fps=VIDEO_FPS)
    print('Saved MP4:', video_path)
    print('Saved GIF:', gif_path)
    display(Image(filename=str(side_paths[0])))


In [ ]:
#@title 9. Per-track sanity report

# Compute diagnostics: duration, displacement, average speed in BEV pixels/frame, jumps, and frame gaps.
rows = []
for tid, g in validation_tracks.sort_values('frame_id').groupby('track_id'):
    g = g.sort_values('frame_id')
    frames = g['frame_id'].to_numpy(dtype=float)
    xs = g['bev_x_px'].to_numpy(dtype=float)
    ys = g['bev_y_px'].to_numpy(dtype=float)
    n = len(g)
    if n >= 2:
        dx = np.diff(xs)
        dy = np.diff(ys)
        df = np.diff(frames)
        step_dist = np.hypot(dx, dy)
        valid_df = np.where(df <= 0, 1, df)
        speed_px_per_frame = step_dist / valid_df
        total_disp = math.hypot(xs[-1] - xs[0], ys[-1] - ys[0])
        max_jump = float(step_dist.max())
        mean_speed = float(speed_px_per_frame.mean())
        max_gap = float(df.max())
    else:
        total_disp = 0.0
        max_jump = 0.0
        mean_speed = 0.0
        max_gap = 0.0
    rows.append({
        'track_id': tid,
        'num_detections': n,
        'first_frame': int(frames.min()),
        'last_frame': int(frames.max()),
        'duration_frames': int(frames.max() - frames.min() + 1),
        'total_displacement_px': total_disp,
        'mean_speed_px_per_frame': mean_speed,
        'max_jump_px': max_jump,
        'max_frame_gap': max_gap,
        'inside_bev_fraction': float(g['inside_bev'].mean()),
    })

track_report = pd.DataFrame(rows).sort_values(['num_detections', 'total_displacement_px'], ascending=False)
report_path = NB06_DIR / 'track_sanity_report.csv'
track_report.to_csv(report_path, index=False)
print('Saved track sanity report:', report_path)
display(track_report.head(30))

# Flag suspicious tracks.
suspicious = track_report[
    (track_report['num_detections'] <= 2) |
    (track_report['max_jump_px'] > 150) |
    (track_report['inside_bev_fraction'] < 0.8)
].copy()
print('Suspicious tracks:', len(suspicious))
display(suspicious.head(30))


In [ ]:
#@title 10. Trajectory plot in BEV with track IDs

traj_img = np.full((BEV_HEIGHT, BEV_WIDTH, 3), 245, dtype=np.uint8)

# Use first warped BEV frame as faint background if available.
first_frame = int(validation_tracks['frame_id'].min())
img_path = get_sample_image_path(first_frame)
if img_path is not None and img_path.exists():
    img = cv2.imread(str(img_path))
    if img is not None:
        bg = cv2.warpPerspective(img, H_image_to_bev, (BEV_WIDTH, BEV_HEIGHT))
        traj_img = cv2.addWeighted(bg, 0.45, traj_img, 0.55, 0)

for tid, g in validation_tracks.sort_values('frame_id').groupby('track_id'):
    g = g.sort_values('frame_id')
    pts = g[['bev_x_px', 'bev_y_px']].to_numpy(dtype=float)
    pts_i = np.round(pts).astype(int)
    color = color_for_id(int(tid))
    for i in range(len(pts_i) - 1):
        p1 = tuple(pts_i[i])
        p2 = tuple(pts_i[i + 1])
        cv2.line(traj_img, p1, p2, color, 2)
    if len(pts_i) > 0:
        cv2.circle(traj_img, tuple(pts_i[0]), 4, color, -1)
        cv2.putText(traj_img, str(int(tid)), tuple(pts_i[0] + np.array([5, -5])), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

traj_path = NB06_DIR / 'bev_track_trajectory_diagnostic.jpg'
cv2.imwrite(str(traj_path), traj_img)
print('Saved trajectory diagnostic:', traj_path)
display(Image(filename=str(traj_path)))


In [ ]:
#@title 11. Summary and next-action hints

summary = {
    'sample_images_count': len(sample_images),
    'validation_rows': int(len(validation_tracks)),
    'unique_frames': int(validation_tracks['frame_id'].nunique()),
    'unique_tracks': int(validation_tracks['track_id'].nunique()),
    'inside_bev_fraction': float(validation_tracks['inside_bev'].mean()),
    'outputs': {
        'frame_mapping': str(mapping_path),
        'validation_tracks_bev': str(validation_path),
        'original_overlays_dir': str(ORIGINAL_OVERLAY_DIR),
        'bev_overlays_dir': str(BEV_OVERLAY_DIR),
        'side_by_side_video': str(NB06_DIR / 'validation_original_vs_bev.mp4'),
        'track_sanity_report': str(report_path),
        'bev_trajectory_diagnostic': str(traj_path),
    }
}

summary_path = NB06_DIR / 'validation_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved validation summary:', summary_path)
print(json.dumps(summary, indent=2))
print('')
print('How to interpret results:')
print('1. If track IDs jump between cars in the original overlay, tracking is the problem.')
print('2. If original boxes are correct but BEV points are offset, homography or ground point choice is the problem.')
print('3. If frame IDs do not match expected source images, frame mapping is the problem.')
print('4. If original and BEV overlays are correct but 3D cuboids are wrong, cuboid export/rendering is the problem.')
